# 실습 8: 통계 + 머신러닝 분류 (1교시)

7주차에서 추출한 23개 특성을 입력으로 사용하여
- **IQR 이상치 탐지** (학습 불필요)
- **RandomForest** (앙상블 결정 트리)
- **XGBoost** (Gradient Boosting)
세 가지 방법의 정확도/속도/해석가능성을 비교합니다.

**사전 조건**: 7주차 `feature_engineering.py` 실행으로 `processed_data.pkl` 생성된 상태


In [4]:
# %% [Setup] 패키지 import 및 데이터 로드
import pickle
import time
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_curve, auc)
import plotly.express as px
import plotly.graph_objects as go

# 7주차 데이터 로드 (상위 폴더의 7_week 참조)
with open(r"C:\Users\6-112\Desktop\prj\bigdata-project\11_week\processed_data.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]
feature_cols = data["feature_cols"]

print(f"학습: {X_train.shape}  /  테스트: {X_test.shape}")
print(f"특성 수: {len(feature_cols)}개")
print(f"테스트 라벨 분포: 정상 {(y_test==0).sum()}건 / 공격 {(y_test==1).sum()}건")


학습: (48852, 23)  /  테스트: (12213, 23)
특성 수: 23개
테스트 라벨 분포: 정상 7200건 / 공격 5013건


## 1. 통계적 방법 — IQR 이상치 탐지

각 특성에서 정상 범위(Q1-1.5·IQR ~ Q3+1.5·IQR)를 벗어난 개수를 합산하여,
임계값 이상이면 공격으로 판별합니다.

In [5]:
# %% [1] IQR 경계 계산 (학습 데이터 기반)
iqr_bounds = []
for i in range(X_train.shape[1]):
    Q1 = np.percentile(X_train[:, i], 25)
    Q3 = np.percentile(X_train[:, i], 75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR if IQR > 0 else None
    upper = Q3 + 1.5 * IQR if IQR > 0 else None
    iqr_bounds.append((lower, upper))

# 테스트 데이터의 각 행이 몇 개 특성에서 이상치인지 카운트
start = time.time()
outlier_scores = np.zeros(len(X_test))
for i, (lower, upper) in enumerate(iqr_bounds):
    if lower is None:
        continue
    is_outlier = (X_test[:, i] < lower) | (X_test[:, i] > upper)
    outlier_scores += is_outlier.astype(int)

threshold = 2  # 2개 이상 특성에서 이상치 → 공격
iqr_pred = (outlier_scores >= threshold).astype(int)
iqr_time = time.time() - start

iqr_acc = accuracy_score(y_test, iqr_pred)
iqr_f1  = f1_score(y_test, iqr_pred)
print(f"IQR 정확도: {iqr_acc:.4f}")
print(f"IQR F1:    {iqr_f1:.4f}")
print(f"IQR 시간:  {iqr_time:.4f}초 ({len(X_test)}건)")


IQR 정확도: 0.6810
IQR F1:    0.5934
IQR 시간:  0.0017초 (12213건)


In [6]:
# %% [2] 이상치 점수 분포 시각화
score_df = pd.DataFrame({
    "score": outlier_scores,
    "label": pd.Series(y_test).map({0:"정상", 1:"공격"}),
})
fig = px.histogram(
    score_df, x="score", color="label",
    title="이상치 점수 분포 — 정상 vs 공격",
    labels={"score":"이상치 점수 (몇 개 특성에서 벗어났는가)"},
    color_discrete_map={"정상":"#22d3ee","공격":"#ef4444"},
    barmode="overlay", opacity=0.7, nbins=24,
)
fig.add_vline(x=threshold, line_dash="dash", line_color="yellow",
              annotation_text=f"임계값 {threshold}")
fig.show()


### IQR 결과의 한계 — "왜 정확도가 낮을까?"

이 히스토그램이 보여주는 **세 가지 본질적 약점** 때문에 IQR 단독 분류는 정확도에 한계가 있습니다.

#### 1. **점수 0인데 공격인 경우가 많음** (미탐 — False Negative)

```
점수 0 막대를 자세히 보면:
  ├─ 정상(cyan):  ~600건 ← 대부분 OK
  └─ 공격(red):   ~200건 ★ 이게 문제!
```

→ 약 **200건의 공격이 23개 특성 모두에서 IQR 범위 안**(=정상 분포 안)에 들어옴
→ IQR로는 절대 못 잡음 — **보안에서 가장 치명적인 미탐**

이런 공격이 발생하는 이유:
- 짧은 SQL Injection (`?id=1` 같은 형태) → URL 길이 정상, 특수문자 적음
- 본문이 짧은 XSS → 길이·엔트로피 모두 정상 범위
- 정상 트래픽과 통계적으로 구분되지 않는 페이로드

#### 2. **점수 1~2 구간이 모호함** (애매한 경계)

```
점수 1: 정상과 공격이 거의 비슷한 비율로 섞임 (~50:50)
점수 2: 임계값 경계라 분류가 불안정
```

→ 임계값을 어디로 잡든 **반드시 오류 발생**
- 임계값 ↓ (예: 1): 공격 더 잡음 (Recall↑) 하지만 정상도 오탐 (Precision↓)
- 임계값 ↑ (예: 3): 정상 보호 강해짐 하지만 공격 더 놓침 (Recall↓)

#### 3. **특성 간 상호작용을 못 봄** — IQR의 본질적 한계

IQR은 **각 특성을 독립적으로** 봅니다:
```
특성 1: url_length 단독 평가 → 정상 범위?
특성 2: has_sql 단독 평가    → 정상 범위?
...
점수 합산만 함
```

하지만 실제로 공격은 **특성 조합**에서 드러납니다:
```
"url_length=80 (정상 범위) + has_sql=1 (이상)"  → 명백한 공격
"url_length=80 + has_sql=0"                    → 정상

IQR은 두 케이스를 구분 못 함 (둘 다 점수 1로 같음)
```

#### 4. **이진 특성(0/1)은 IQR 적용 불가**

```python
if IQR == 0:
    continue   # 분산 없는 특성은 건너뜀
```

- `has_sql`, `has_xss`, `has_traversal` 같은 **이진 특성**은 Q1=0, Q3=1 또는 모두 0 → IQR=0
- → 이런 특성들은 **계산에서 빠짐**
- 하지만 이 특성들은 **공격 탐지의 가장 강력한 신호**!
- IQR이 가장 유용한 정보를 활용 못 하는 셈

---

### 그래서 머신러닝이 필요하다

IQR의 위 4가지 한계를 모두 해결하는 모델이 **RandomForest**입니다:

| IQR의 한계 | RandomForest의 해결 |
|---|---|
| ① 점수 0인 공격 미탐 | 분기 트리가 미세한 특성 조합도 포착 |
| ② 모호한 경계 | 100그루 다수결로 안정적 판단 |
| ③ 특성 독립 평가 | 분기 흐름이 **특성 간 상호작용** 학습 |
| ④ 이진 특성 무시 | `has_sql == 1?` 같은 분기에 적극 활용 |

→ 다음 단계에서 RandomForest로 학습해 **얼마나 개선되는지** 직접 확인합니다.

## 2. 머신러닝 — RandomForest

100개의 결정 트리를 만들어 다수결로 분류합니다.
`class_weight="balanced"`로 정상/공격 불균형을 보정합니다.

In [7]:
# %% [3] RandomForest 학습 + 평가
from sklearn.ensemble import RandomForestClassifier

start = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_train_time = time.time() - start

start = time.time()
rf_pred = rf.predict(X_test)
rf_pred_time = time.time() - start

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1  = f1_score(y_test, rf_pred)
print(f"RF 정확도:  {rf_acc:.4f}")
print(f"RF F1:     {rf_f1:.4f}")
print(f"학습 시간: {rf_train_time:.2f}초")
print(f"예측 시간: {rf_pred_time:.4f}초 ({len(X_test)}건)")
print()
print(classification_report(y_test, rf_pred, target_names=["Normal","Anomalous"]))


RF 정확도:  0.9274
RF F1:     0.9108
학습 시간: 0.39초
예측 시간: 0.0389초 (12213건)

              precision    recall  f1-score   support

      Normal       0.93      0.94      0.94      7200
   Anomalous       0.92      0.90      0.91      5013

    accuracy                           0.93     12213
   macro avg       0.93      0.92      0.92     12213
weighted avg       0.93      0.93      0.93     12213



In [8]:
# %% [4] 특성 중요도 Top 10
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False).head(10)

fig = px.bar(
    importance_df, x="importance", y="feature", orientation="h",
    title="RandomForest 특성 중요도 Top 10",
    color="importance", color_continuous_scale="Viridis",
)
fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=420)
fig.show()
print(importance_df.to_string(index=False))


           feature  importance
       upper_ratio    0.213016
       digit_ratio    0.151588
      total_length    0.113625
       url_entropy    0.101339
special_char_ratio    0.072066
          num_dots    0.058231
       num_slashes    0.057938
        url_length    0.049561
        num_spaces    0.044059
 num_special_chars    0.031476


## 3. 머신러닝 — XGBoost (심화)

이전 트리의 오답을 다음 트리가 보정하는 방식(Gradient Boosting).
RandomForest보다 약간 더 정확한 경우가 많습니다.

In [10]:
# %% [5] XGBoost 학습 + 평가
from xgboost import XGBClassifier

# 클래스 불균형 보정 (옵션 A — class_weight 대응)
# XGBoost는 class_weight가 없으므로 scale_pos_weight 사용
#   = 음성 클래스 수 / 양성 클래스 수
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos if pos > 0 else 1.0
print(f"scale_pos_weight = {spw:.3f}  (Normal {neg:,} / Anomalous {pos:,})")

start = time.time()
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric="logloss",
    scale_pos_weight=spw,   # ★ 클래스 불균형 자동 보정
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, y_train)
xgb_train_time = time.time() - start

start = time.time()
xgb_pred = xgb.predict(X_test)
xgb_pred_time = time.time() - start

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1  = f1_score(y_test, xgb_pred)
print(f"XGB 정확도: {xgb_acc:.4f}")
print(f"XGB F1:    {xgb_f1:.4f}")
print(f"학습 시간: {xgb_train_time:.2f}초")
print(f"예측 시간: {xgb_pred_time:.4f}초")

scale_pos_weight = 1.436  (Normal 28,800 / Anomalous 20,052)
XGB 정확도: 0.9257
XGB F1:    0.9121
학습 시간: 0.26초
예측 시간: 0.0048초


## 4. 평가 지표 — 혼동행렬 + ROC

정확도만 보면 안 되는 이유: 불균형 데이터에서는 **F1 + 재현율**이 핵심.
보안에서는 미탐(FN)이 오탐(FP)보다 훨씬 위험합니다.

In [11]:
# %% [6] 혼동행렬 (RandomForest 기준)
cm = confusion_matrix(y_test, rf_pred)
fig = px.imshow(
    cm, text_auto=True,
    labels={"x":"예측","y":"실제"},
    x=["정상","공격"], y=["정상","공격"],
    color_continuous_scale="Blues",
    title="RandomForest 혼동행렬",
)
fig.show()

TN, FP, FN, TP = cm.ravel()
print(f"TN={TN}, FP={FP}(오탐), FN={FN}(미탐 ★ 위험), TP={TP}")
print(f"재현율(Recall): {TP/(TP+FN):.4f}  ← 보안에서 중시")
print(f"정밀도(Precision): {TP/(TP+FP):.4f}")


TN=6800, FP=400(오탐), FN=487(미탐 ★ 위험), TP=4526
재현율(Recall): 0.9029  ← 보안에서 중시
정밀도(Precision): 0.9188


In [12]:
# %% [7] ROC Curve
rf_proba = rf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, rf_proba)
roc_auc = auc(fpr, tpr)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                         name=f"RF (AUC={roc_auc:.3f})", line=dict(color="#22d3ee")))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                         line=dict(dash="dash", color="gray"), name="Random"))
fig.update_layout(title="ROC Curve",
                  xaxis_title="False Positive Rate",
                  yaxis_title="True Positive Rate")
fig.show()


## 5. 세 방법 비교 + 결과 저장 (2교시 LLM과 비교용)

In [13]:
# %% [8] 비교 표 + 저장
results = pd.DataFrame([
    {"방법":"IQR (통계)",   "정확도":iqr_acc, "F1":iqr_f1, "학습 필요":"X",
     "예측 시간(s)":iqr_time},
    {"방법":"RandomForest", "정확도":rf_acc,  "F1":rf_f1,  "학습 필요":"O",
     "예측 시간(s)":rf_pred_time},
    {"방법":"XGBoost",      "정확도":xgb_acc, "F1":xgb_f1, "학습 필요":"O",
     "예측 시간(s)":xgb_pred_time},
])
print(results.to_string(index=False))

# 2교시 비교에 사용할 결과 저장
with open("classification_results.pkl", "wb") as f:
    pickle.dump({
        "rf_model": rf,
        "xgb_model": xgb,
        "iqr_bounds": iqr_bounds,
        "iqr_pred": iqr_pred,
        "rf_pred": rf_pred,
        "xgb_pred": xgb_pred,
        "y_test": y_test,
        "feature_cols": feature_cols,
        "results": results,
    }, f)
print("\n>> classification_results.pkl 저장 완료")


          방법      정확도       F1 학습 필요  예측 시간(s)
    IQR (통계) 0.680996 0.593404     X  0.001681
RandomForest 0.927372 0.910756     O  0.038902
     XGBoost 0.925653 0.912101     O  0.004804

>> classification_results.pkl 저장 완료


## 핵심 정리

| 방법 | 정확도 | 속도 | 학습 | 해석 |
|------|--------|------|------|------|
| IQR | 낮음 | 매우 빠름 | X | 임계값 |
| RF | 높음 | 빠름 | O | 특성 중요도 |
| XGBoost | 매우 높음 | 보통 | O | 특성 중요도 |

**다음**: 2교시에서 같은 데이터를 **Ollama LLM**으로 분류하고 비교합니다.